In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score
from yellowbrick.classifier import ConfusionMatrix
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif


In [3]:
credito = pd.read_csv("dados/Credit.csv")
credito.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,'critical/other existing credit',radio/tv,1169,'no known savings',>=7,4,'male single',none,...,'real estate',67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,'existing paid',radio/tv,5951,<100,1<=X<4,2,'female div/dep/mar',none,...,'real estate',22,none,own,1,skilled,1,none,yes,bad
2,'no checking',12,'critical/other existing credit',education,2096,<100,4<=X<7,2,'male single',none,...,'real estate',49,none,own,1,'unskilled resident',2,none,yes,good
3,<0,42,'existing paid',furniture/equipment,7882,<100,4<=X<7,2,'male single',guarantor,...,'life insurance',45,none,'for free',1,skilled,2,none,yes,good
4,<0,24,'delayed previously','new car',4870,<100,1<=X<4,3,'male single',none,...,'no known property',53,none,'for free',2,skilled,2,none,yes,bad


In [4]:
credito.tail(20)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
980,0<=X<200,30,'critical/other existing credit',furniture/equipment,8386,<100,4<=X<7,2,'male single',none,...,'life insurance',49,none,own,1,skilled,1,none,yes,bad
981,'no checking',48,'existing paid',business,4844,<100,unemployed,3,'male single',none,...,car,33,bank,rent,1,'high qualif/self emp/mgmt',1,yes,yes,bad
982,>=200,21,'existing paid','new car',2923,100<=X<500,1<=X<4,1,'female div/dep/mar',none,...,car,28,bank,own,1,'high qualif/self emp/mgmt',1,yes,yes,good
983,<0,36,'existing paid','used car',8229,<100,1<=X<4,2,'male single',none,...,'life insurance',26,none,own,1,skilled,2,none,yes,bad
984,'no checking',24,'critical/other existing credit',furniture/equipment,2028,<100,4<=X<7,2,'male single',none,...,'life insurance',30,none,own,2,'unskilled resident',1,none,yes,good
985,<0,15,'critical/other existing credit',furniture/equipment,1433,<100,1<=X<4,4,'female div/dep/mar',none,...,'life insurance',25,none,rent,2,skilled,1,none,yes,good
986,>=200,42,'no credits/all paid',business,6289,<100,<1,2,'male div/sep',none,...,'life insurance',33,none,own,2,skilled,1,none,yes,good
987,'no checking',13,'existing paid',radio/tv,1409,100<=X<500,unemployed,2,'female div/dep/mar',none,...,'real estate',64,none,own,1,skilled,1,none,yes,good
988,<0,24,'existing paid','used car',6579,<100,unemployed,4,'male single',none,...,'no known property',29,none,'for free',1,'high qualif/self emp/mgmt',1,yes,yes,good
989,0<=X<200,24,'critical/other existing credit',radio/tv,1743,<100,>=7,4,'male single',none,...,'life insurance',48,none,own,2,'unskilled resident',1,none,yes,good


In [5]:
credito.shape

(1000, 21)

In [ ]:
#Separar os tipos de coluna entre: numéricos, categóricos ordinais e categóricos nominais

# Separar a variável que representa o resultado (y) das variáveis que são só características (X)
X = credito.drop(columns=['class'])
y = credito['class']

# 1. armazenando apenas as colunas numéricas, as que usam números inteiros e números float
colunas_numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 2. armazenando as colunas categóricas, utilizando essas classes que indicam textos.
colunas_categoricas = X.select_dtypes(include=['string','object', 'category']).columns.tolist()

print("Numéricas:", colunas_numericas)
print("Categóricas:", colunas_categoricas)

Numéricas: ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Categóricas: ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']


In [7]:
#Separar as categóricas entre Ordinais e Nominais
colunas_ordinais = ['checking_status', 'credit_history' , 'savings_status', 'employment', 'property_magnitude' ]
colunas_nominais = [col for col in colunas_categoricas if col not in colunas_ordinais]

print("Nominais:", colunas_nominais)
print("Ordinais:", colunas_ordinais)
print("Numéricas:", colunas_numericas)

Nominais: ['purpose', 'personal_status', 'other_parties', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']
Ordinais: ['checking_status', 'credit_history', 'savings_status', 'employment', 'property_magnitude']
Numéricas: ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']


In [8]:
#Analisando se há a falta de informação em alguma parte do banco de dados.
credito.isnull().sum()
#no caso, não há

checking_status           0
duration                  0
credit_history            0
purpose                   0
credit_amount             0
savings_status            0
employment                0
installment_commitment    0
personal_status           0
other_parties             0
residence_since           0
property_magnitude        0
age                       0
other_payment_plans       0
housing                   0
existing_credits          0
job                       0
num_dependents            0
own_telephone             0
foreign_worker            0
class                     0
dtype: int64

In [9]:
#Calcular a proporção entre os resultados da classe (y).Isso é feito para evitar uma desregulação no resultado da IA preditiva e buscar uma porcentagem
#correspondente ao real resultado esperado.
proporcao = credito['class'].value_counts(normalize=True) * 100
print(proporcao)
#A accuracy da IA sem análise acerta 71% das vezes, mas como 70% da classe é "good", então ela só precisa afirmar que todas são "good", e terá 70% de acerto.

class
good    70.0
bad     30.0
Name: proportion, dtype: float64


In [16]:
#Começa a análise de relevância das colunas categóricas, utilizando o teste do Qui-Quadrado. O teste do Qui-Quadrado é utilizado para verificar se existe uma associação significativa entre duas variáveis categóricas. No caso, queremos verificar se cada coluna categórica tem alguma relação com a variável alvo 'classe'.
#Loop passando por todas as colunas categóricas

colunas_descartar = []

for coluna in colunas_categoricas:
    tabela_cruzada = pd.crosstab(credito[coluna], credito['class'])
    
    stat, p_valor, dof, esperados = chi2_contingency(tabela_cruzada)
    
    # Verifica se o p-valor é MAIOR que 0.05 (5%)
    if p_valor > 0.05:
        colunas_descartar.append(coluna)
        print(f" Coluna '{coluna}': p-value = {p_valor:.4f} ")

print(f"\nTotal de colunas que podem ser descartadas: {len(colunas_descartar)}")
print("Lista de colunas com p-value > 0.05:", colunas_descartar)

 Coluna 'job': p-value = 0.5966 
 Coluna 'own_telephone': p-value = 0.2789 

Total de colunas que podem ser descartadas: 2
Lista de colunas com p-value > 0.05: ['job', 'own_telephone']


In [ ]:
#Mdificando o dataframe original, descartando as colunas que não são relevantes para a análise.
credito_modificado1 = credito.drop(columns=colunas_descartar)
print(credito_modificado1.shape)

(1000, 19)


In [23]:

# 1. Separar atributos (X) e classe alvo (y) do seu dataset modificado
X = credito_modificado1.drop(columns=['class'])
y = credito_modificado1['class']

# 2. Criar a variável X_encoded convertendo textos/categorias em números (One-Hot Encoding)
X_encoded = pd.get_dummies(X, drop_first=True)


In [24]:
#Utilizando a Informação Mútua para analisar a relevância das colunas numéricas. A Informação Mútua é uma medida de dependência entre duas variáveis. No caso, queremos verificar se cada coluna numérica tem alguma relação com a variável alvo 'classe'.

#Cruzar os dados com a variavel alvo para dexcobrir a relevancia de cada uma das colunas.
importancia = mutual_info_classif(X_encoded, y, discrete_features=True)

credito_modificado1_im = pd.DataFrame({'Atributo': X_encoded.columns, 'Informacao_Mutua': importancia})
credito_modificado1_im = credito_modificado1_im.sort_values(by='Informacao_Mutua', ascending=False)
print(credito_modificado1_im)

                                           Atributo  Informacao_Mutua
1                                     credit_amount      5.710016e-01
0                                          duration      4.391406e-02
8                                checking_status_<0      3.186429e-02
4                                               age      3.159751e-02
10  credit_history_'critical/other existing credit'      1.768567e-02
25                              savings_status_<100      1.334965e-02
13             credit_history_'no credits/all paid'      9.409776e-03
41                                      housing_own      8.802293e-03
36           property_magnitude_'no known property'      7.492300e-03
37                 property_magnitude_'real estate'      7.423682e-03
7                          checking_status_0<=X<200      6.945457e-03
39                         other_payment_plans_none      6.149269e-03
20                                 purpose_radio/tv      5.934079e-03
15                  

In [25]:
#Organizar a lista para mostrar os números mais próximos de 0

pd.set_option('display.float_format', lambda x: '%.4f' % x)

df_mi_organizado = credito_modificado1_im.copy()
df_mi_organizado['Informacao_Mutua'] = df_mi_organizado['Informacao_Mutua'].round(4)
df_mi_organizado = df_mi_organizado.reset_index(drop=True)

# Exibe o resultado organizado
df_mi_organizado

,Atributo,Informacao_Mutua
0,credit_amount,0.5710
1,duration,0.0439
2,checking_status_<0,0.0319
3,age,0.0316
4,credit_history_'critical/other existing credit',0.0177
5,savings_status_<100,0.0133
6,credit_history_'no credits/all paid',0.0094
7,housing_own,0.0088
8,property_magnitude_'no known property',0.0075
9,property_magnitude_'real estate',0.0074
